# Ensemble de três arquiteturas com validação cruzada

Este caderno reúne as predições da DenseNet169, DenseNet201 e EfficientNetB4. Cada arquitetura é avaliada com os mesmos cinco folds e com oversampling restrito ao conjunto de treino. Ao final, as probabilidades out-of-fold são combinadas por *soft voting*.

Os três modelos recebem o mesmo peso. Essa escolha foi feita antes da avaliação e não depende do desempenho observado nos folds. Desse modo, as métricas do ensemble continuam sendo uma estimativa out-of-fold, sem ajuste de pesos sobre as amostras usadas na avaliação. O conjunto `test_real_split.csv` permanece reservado para a etapa final do trabalho.

## 1. Preparação do ambiente

A célula abaixo localiza o repositório mesmo quando o notebook é aberto a partir de outra pasta.

In [ ]:
import sys
from pathlib import Path

project_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "03_transfer_learning").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError("Não foi possível localizar a raiz do repositório.")

architecture_module_dir = project_root / "03_transfer_learning" / "cross_validation"
ensemble_module_dir = project_root / "04_ensembles" / "cross_validation"
for module_dir in (architecture_module_dir, ensemble_module_dir):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from cross_validation_utils import CrossValidationConfig, run_cross_validation
from ensemble_cross_validation_utils import (
    load_aligned_predictions,
    run_soft_voting_ensemble,
)

print("Repositório:", project_root)

## 2. Configuração dos modelos

As configurações são as mesmas usadas nos notebooks individuais. A seed comum garante que as três arquiteturas sejam avaliadas exatamente nas mesmas divisões.

In [ ]:
model_configs = [
    CrossValidationConfig(
        architecture="densenet169",
        image_size=(320, 320),
        batch_size=16,
        epochs=60,
        n_splits=5,
        learning_rate=1e-4,
        early_stopping_patience=8,
        reduce_lr_patience=4,
        seed=42,
        balance_strategy="random_oversampling",
        backbone_weights="imagenet",
        save_fold_models=False,
    ),
    CrossValidationConfig(
        architecture="densenet201",
        image_size=(320, 320),
        batch_size=12,
        epochs=60,
        n_splits=5,
        learning_rate=1e-4,
        early_stopping_patience=8,
        reduce_lr_patience=4,
        seed=42,
        balance_strategy="random_oversampling",
        backbone_weights="imagenet",
        save_fold_models=False,
    ),
    CrossValidationConfig(
        architecture="efficientnetb4",
        image_size=(380, 380),
        batch_size=8,
        epochs=60,
        n_splits=5,
        learning_rate=1e-4,
        early_stopping_patience=8,
        reduce_lr_patience=4,
        seed=42,
        balance_strategy="random_oversampling",
        backbone_weights="imagenet",
        save_fold_models=False,
    ),
]

model_slugs = [config.slug for config in model_configs]
model_slugs

## 3. Treinamento das arquiteturas

A execução completa corresponde a quinze treinamentos: cinco para cada arquitetura. Como esse processo é demorado, o notebook verifica se `oof_predictions.csv` já existe e reaproveita o resultado concluído. Para repetir um experimento, defina `SKIP_COMPLETED = False`.

In [ ]:
SKIP_COMPLETED = True
results_root = project_root / "results" / "cross_validation"

for config in model_configs:
    predictions_path = results_root / config.slug / "oof_predictions.csv"
    if SKIP_COMPLETED and predictions_path.is_file():
        print(f"{config.architecture}: resultado existente, treinamento ignorado.")
        continue

    print(f"{config.architecture}: iniciando validação cruzada.")
    run_cross_validation(config)

## 4. Conferência das predições

Antes de calcular o ensemble, são comparados os identificadores das imagens, os rótulos e o número do fold. A combinação só prossegue se os três arquivos estiverem alinhados.

In [ ]:
aligned_predictions = load_aligned_predictions(model_slugs)

for model_slug, dataframe in aligned_predictions.items():
    print(f"{model_slug}: {len(dataframe)} predições OOF")

## 5. Cálculo do ensemble

O *soft voting* calcula a média das sete probabilidades produzidas pelos modelos. A classe com maior probabilidade média é adotada como resposta final.

In [ ]:
ensemble_results = run_soft_voting_ensemble(
    model_slugs,
    weights=[1, 1, 1],
    ensemble_slug="ensemble-3-soft-voting",
)

print("Resultados salvos em:", ensemble_results["output_dir"])

## 6. Resultados

A primeira tabela permite comparar o ensemble com cada arquitetura. A segunda apresenta a variação das métricas entre os cinco folds.

In [ ]:
display(ensemble_results["model_comparison"])
display(ensemble_results["fold_metrics"])
display(ensemble_results["summary_metrics"])